# Tutorial 12: Universal Graph Neural Networks for Quantum Circuit Design

## Why Graph ML?

Tutorial 8 trained a tabular DNN: design parameters → Hamiltonian targets. This works for a **fixed** topology but breaks if you change the circuit structure.

The **Universal GNN** replaces this with a **heterogeneous graph** of geometric embeddings:
1. Design parameters → `build_layout()` → Shapely polygons
2. Components → **static embedding** = `param_sum ∥ geometric_moments ∥ shape_tensor`
3. Connections → **typed physical edges** with coupling/overlap geometry
4. Full layout → **virtual hub node** for global context
5. **HeteroConv GNN** learns which design parameters affect which Hamiltonian targets

### Target assignment
- **Node targets**: `qubit_freq`, `anharmonicity`, `cavity_freq` — intrinsic to components
- **Edge targets**: `g` (coupling strength), `kappa` (resonator linewidth) — arise from interactions

During training, ALL nodes get ALL node targets and ALL edges get ALL edge targets. The GNN discovers the correlations.
At inference, we use a **readout map** to extract predictions from the correct nodes/edges.


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from shapely.affinity import translate, scale as shp_scale
from shapely.geometry import MultiPolygon, box
from shapely.ops import unary_union
from sklearn.metrics import r2_score
from torch_geometric.loader import DataLoader

from squadds.ml.universal import (
    EDGE_INFERENCE_READOUT,
    EDGE_TARGET_NAMES,
    NODE_INFERENCE_READOUT,
    NODE_TARGET_NAMES,
    CircuitNetlist,
    ComponentSpec,
    EdgeSpec,
    EdgeFeatureExtractor,
    EmbeddingConfig,
    EmbeddingMode,
    EmbeddingVersion,
    UniversalGraphBuilder,
    UniversalTrainer,
    benchmark_component_family_clustering,
    benchmark_standard_embedding_arithmetic,
    build_component_embedding_collection,
    build_graph_dataset,
    build_graph_from_row,
    build_layout,
    build_layout_from_row,
    build_model_from_graph,
    compute_component_embedding,
    compute_cosine_similarity_matrix,
    compute_embedding_projections,
    edge_feature_dim,
    embedding_dim,
    evaluate_standard_arithmetic_case,
    find_nearest_neighbors,
    get_polygon_for_component,
    make_standard_qubit_cavity_netlist,
    param_feature_dim,
    plot_component,
    plot_layout,
    plot_projection_grid,
    plot_similarity_bars,
    read_prediction_summary,
    STANDARD_ARITHMETIC_SPECS,
)
from squadds.ml.universal.features.moments import compute_moments, moment_names
from squadds.ml.universal.features.node_encoder import DEFAULT_SHAPE_RESOLUTION

SHAPE_RES = DEFAULT_SHAPE_RESOLUTION * 6
EMBEDDING_CONFIG = EmbeddingConfig(
    version=EmbeddingVersion.V2_HASHED_PARAMS,
    mode=EmbeddingMode.GEOMETRY_ONLY,
    shape_resolution=SHAPE_RES,
    param_hash_dim=8,
)
GRAPH_GLOBAL_FEATURES = {"dielectric_constant": 11.45}

print(f"Shape resolution: {SHAPE_RES}x{SHAPE_RES}")
print(f"Embedding config: {EMBEDDING_CONFIG.to_metadata()}")
print(f"Node embedding dim: {embedding_dim(EMBEDDING_CONFIG)}")
print(f"Parameter feature dim: {param_feature_dim(EMBEDDING_CONFIG)}")
print(f"Edge feature dim: {edge_feature_dim(SHAPE_RES)}")
print(f"Node targets: {NODE_TARGET_NAMES}")
print(f"Edge targets: {EDGE_TARGET_NAMES}")

seed = 42
torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)

---\n## 1. From Design Parameters to Physical Layout

In [ ]:
df = pd.read_parquet("data/training_data.parquet").drop_duplicates().reset_index(drop=True)
print(f"Dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head(3)


In [ ]:
row = df.iloc[0]
lyt = build_layout(
    cross_length=row["cross_length"], cross_gap=row["cross_gap"],
    claw_length=row["claw_length"], ground_spacing=row["ground_spacing"],
    coupling_length=row["coupling_length"], total_length=row["total_length"],
)
fig = plot_layout(lyt)
plt.suptitle(f"Layout: cross_length={row['cross_length']}, total_length={row['total_length']}", y=1.01)
plt.show()


---\n## 2. Static Embedding Deep Dive

In [ ]:
comp_names = ["qubit", "claw", "resonator", "feedline"]
param_dim = param_feature_dim(EMBEDDING_CONFIG)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, name in enumerate(comp_names):
    poly = get_polygon_for_component(lyt[name])
    params = lyt[name].get("params", {})
    emb = compute_component_embedding(poly, params=params, config=EMBEDDING_CONFIG)
    mom = compute_moments(poly)
    print(f"=== {name.upper()} ===")
    print(f"  param_features[:6] = {np.round(emb[:min(6, param_dim)], 2)}")
    for mn, mv in zip(moment_names(), mom):
        print(f"  {mn:20s}: {mv:12.2f}")
    print()

    axes[0, i].imshow(emb[param_dim + 8 :].reshape(SHAPE_RES, SHAPE_RES), cmap="viridis", interpolation="nearest")
    axes[0, i].set_title(f"{name.title()} Shape Tensor")
    axes[0, i].axis("off")
    axes[1, i].barh(moment_names(), mom, color="steelblue")
    axes[1, i].set_title(f"{name.title()} Moments")
    axes[1, i].tick_params(labelsize=7)

plt.suptitle("Static Embeddings (versioned protocol)", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### Embedding Space, Clustering Benchmark, and Neighborhood Analysis

In [ ]:
benchmark_rows = df.head(200).to_dict("records")
collection = build_component_embedding_collection(
    benchmark_rows,
    embedding_config=EMBEDDING_CONFIG,
)
cluster_benchmark = benchmark_component_family_clustering(collection.embeddings, collection.labels)

display(
    pd.DataFrame(
        [
            {
                "num_embeddings": cluster_benchmark.num_embeddings,
                "num_labels": cluster_benchmark.num_labels,
                "centroid_top1_accuracy": cluster_benchmark.centroid_top1_accuracy,
                "nearest_neighbor_top1_accuracy": cluster_benchmark.nearest_neighbor_top1_accuracy,
                "mean_intra_label_similarity": cluster_benchmark.mean_intra_label_similarity,
                "mean_inter_label_similarity": cluster_benchmark.mean_inter_label_similarity,
                "separation_gap": cluster_benchmark.separation_gap,
            }
        ]
    )
)

display(
    pd.DataFrame(
        [
            {
                "label": item.label,
                "count": item.count,
                "mean_self_centroid_similarity": item.mean_self_centroid_similarity,
                "mean_nearest_other_centroid_similarity": item.mean_nearest_other_centroid_similarity,
                "separation_margin": item.separation_margin,
            }
            for item in cluster_benchmark.per_label
        ]
    )
)

projection_titles = {
    "pca": "PCA\n(linear, global variance)",
    "kernel_pca": "Kernel PCA (RBF)\n(non-linear, global)",
    "tsne": "t-SNE\n(non-linear, local structure)",
    "umap": "UMAP\n(local + global)",
}
projections = compute_embedding_projections(
    collection.embeddings,
    methods=("pca", "kernel_pca", "tsne", "umap"),
    random_state=42,
    method_kwargs={
        "tsne": {"perplexity": 30},
        "umap": {"n_neighbors": 20, "min_dist": 0.15},
    },
)
palette = {
    "Qubit": "#2196F3",
    "Claw": "#FF9800",
    "Resonator": "#4CAF50",
    "Feedline": "#9C27B0",
}

fig, _ = plot_projection_grid(
    {projection_titles[name]: projection for name, projection in projections.items()},
    collection.labels,
    palette=palette,
    figsize=(24, 6),
    suptitle="Component Embedding Space — maintained projection APIs",
)
plt.show()

query_index = next(i for i, label in enumerate(collection.labels) if label == "Qubit")
neighbors = find_nearest_neighbors(
    collection.embeddings[query_index],
    collection.embeddings,
    labels=collection.labels,
    identifiers=collection.identifiers,
    top_k=5,
    exclude_index=query_index,
)
display(
    pd.DataFrame(
        [
            {
                "identifier": item.identifier,
                "label": item.label,
                "cosine_similarity": item.similarity,
            }
            for item in neighbors
        ]
    )
)

subset = np.arange(min(40, len(collection.embeddings)))
sim = compute_cosine_similarity_matrix(collection.embeddings[subset])
plt.figure(figsize=(8, 6))
sns.heatmap(sim, cmap="viridis", square=True, cbar_kws={"label": "Cosine similarity"})
plt.title("Cosine similarity matrix (first 40 embeddings)", fontweight="bold")
plt.xlabel("Embedding index")
plt.ylabel("Embedding index")
plt.tight_layout()
plt.show()

### Embedding Arithmetic Benchmark and Single-Case Inspection

In [ ]:
arithmetic_rows = df.head(120).to_dict("records")
arithmetic_benchmark = benchmark_standard_embedding_arithmetic(
    arithmetic_rows,
    shape_resolution=SHAPE_RES,
)

display(
    pd.DataFrame(
        [
            {
                "num_trials": arithmetic_benchmark.num_trials,
                "top1_accuracy": arithmetic_benchmark.top1_accuracy,
                "top2_accuracy": arithmetic_benchmark.top2_accuracy,
                "mean_expected_rank": arithmetic_benchmark.mean_expected_rank,
                "mean_expected_similarity": arithmetic_benchmark.mean_expected_similarity,
                "mean_margin": arithmetic_benchmark.mean_margin,
            }
        ]
    )
)

display(
    pd.DataFrame(
        [
            {
                "case_name": item.case_name,
                "expected_label": item.expected_label,
                "num_trials": item.num_trials,
                "top1_accuracy": item.top1_accuracy,
                "top2_accuracy": item.top2_accuracy,
                "mean_expected_rank": item.mean_expected_rank,
                "mean_expected_similarity": item.mean_expected_similarity,
                "mean_margin": item.mean_margin,
            }
            for item in arithmetic_benchmark.per_case
        ]
    )
)

demo_trial = max(
    [trial for trial in arithmetic_benchmark.trials if trial.case_name == STANDARD_ARITHMETIC_SPECS[0].name],
    key=lambda item: item.margin,
)
demo_row = df.iloc[demo_trial.row_index]
demo_layout = build_layout_from_row(demo_row.to_dict())
demo_qubit = get_polygon_for_component(demo_layout["qubit"])
demo_claw = get_polygon_for_component(demo_layout["claw"])
demo_union = unary_union([demo_qubit, demo_claw])
demo_diff = demo_union.difference(demo_claw)

def draw_polygon(ax, polygon, title, color):
    polygons = polygon.geoms if isinstance(polygon, MultiPolygon) else [polygon]
    for geom in polygons:
        ax.fill(*geom.exterior.xy, alpha=0.65, color=color)
        ax.plot(*geom.exterior.xy, color=color, linewidth=1.4)
    ax.set_title(title, fontweight="bold")
    ax.set_aspect("equal")
    ax.axis("off")

fig, axes = plt.subplots(1, 4, figsize=(19, 4.5))
draw_polygon(axes[0], demo_union, "Union: qubit + claw", "#6a4c93")
draw_polygon(axes[1], demo_claw, "Subtrahend: claw", "#ff9800")
draw_polygon(axes[2], demo_diff, "Difference: (qubit + claw) - claw", "#2196f3")
plot_similarity_bars(demo_trial.matches, ax=axes[3], title="Difference-vector ranking", color="#3a86ff")
plt.tight_layout()
plt.show()

print(f"Selected row index: {demo_trial.row_index}")
print(f"Case: {demo_trial.case_name}")
print(f"Expected label: {demo_trial.expected_label}")
print(f"Predicted label: {demo_trial.predicted_label}")
print(f"Expected rank: {demo_trial.expected_rank}")

---
## 3. Heterogeneous Graph Assembly

| Nodes | Edges |
|---|---|
| `component` (static embedding) | `physical` (component ↔ component): coupling + overlap geometry |
| `virtual` (full layout embedding) | `spatial_{in,out}` (component ↔ virtual): rel pos + area/perim fractions |

**Node targets** (ALL nodes predict ALL during training):
| Target | Physically from | Read at inference from |
|---|---|---|
| `qubit_freq_GHz` | TransmonCross | TransmonCross node |
| `anharmonicity_MHz` | TransmonCross | TransmonCross node |
| `cavity_freq_GHz` | RouteMeander | RouteMeander node |

**Edge targets** (ALL edges predict ALL during training):
| Target | Physically from | Read at inference from |
|---|---|---|
| `g_MHz` | qubit-claw interaction | TransmonCross↔Claw edge |
| `kappa_kHz` | resonator-feedline interaction | RouteMeander↔CoupledLineTee edge |


In [ ]:
netlist = make_standard_qubit_cavity_netlist()
builder = UniversalGraphBuilder(
    shape_resolution=SHAPE_RES,
    cache_dir="graph_cache",
    embedding_config=EMBEDDING_CONFIG,
)
data = build_graph_from_row(
    row.to_dict(),
    netlist=netlist,
    builder=builder,
    global_features=GRAPH_GLOBAL_FEATURES,
    include_targets=False,
)
print("=== HeteroData Graph ===")
print(data)

print("\nNode inference readout:")
for name, ctype, readout in zip(
    data["component"].component_name,
    data["component"].component_type,
    data["component"].inference_readout,
):
    print(f"  {name:12s} ({ctype:16s}): {readout if readout else '(passive)'}")

print("\nEdge inference readout:")
for i, (src_t, dst_t) in enumerate(data["component", "physical", "component"].edge_component_types):
    readout = EDGE_INFERENCE_READOUT.get((src_t, dst_t), []) or EDGE_INFERENCE_READOUT.get((dst_t, src_t), [])
    if readout:
        print(f"  edge {i}: {src_t} <-> {dst_t} -> reads {readout}")

In [ ]:
# Edge overlap + hub masked shapes
edge_extractor = EdgeFeatureExtractor(shape_resolution=SHAPE_RES)
epairs = [("qubit","claw","cap"),("claw","resonator","galv"),("resonator","feedline","cap")]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, (s, d, _) in enumerate(epairs):
    feat = edge_extractor.extract(get_polygon_for_component(lyt[s]),
        get_polygon_for_component(lyt[d]), coupling_type=["capacitive","galvanic","capacitive"][i])
    axes[i].imshow(feat[8:].reshape(SHAPE_RES, SHAPE_RES), cmap='hot', interpolation='nearest')
    axes[i].set_title(f"{s} <-> {d}"); axes[i].axis('off')
plt.suptitle("Physical Edge: Overlap Shape Tensors", fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


---
## 4. Training

Each row → HeteroData graph. ALL nodes get ALL 3 node targets. ALL edges get ALL 2 edge targets.
The GNN learns the physics through message passing.


In [ ]:
N_SAMPLES = 5000  # Increase for production

df_sub = df.head(N_SAMPLES)
rows_sub = df_sub.to_dict("records")
builder = UniversalGraphBuilder(
    shape_resolution=SHAPE_RES,
    cache_dir="graph_cache",
    embedding_config=EMBEDDING_CONFIG,
)

print(f"Building {N_SAMPLES} graphs through workflow helpers...")
graph_dataset = build_graph_dataset(
    rows_sub,
    netlist=netlist,
    builder=builder,
    global_features=GRAPH_GLOBAL_FEATURES,
)

print(f"Done: {len(graph_dataset)} graphs")
print(f"Node targets shape: {graph_dataset[0]['component'].y.shape} (all nodes, 3 targets)")
print(
    f"Edge targets shape: {graph_dataset[0]['component','physical','component'].y.shape} "
    "(all edges, 2 targets)"
)

In [ ]:
split = int(0.85 * len(graph_dataset))
train_loader = DataLoader(graph_dataset[:split], batch_size=32, shuffle=True)
val_loader = DataLoader(graph_dataset[split:], batch_size=32)

model = build_model_from_graph(
    graph_dataset[0],
    hidden_dim=128,
    num_layers=3,
    num_heads=4,
    edge_hidden=32,
)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

trainer = UniversalTrainer(model, learning_rate=1e-3, checkpoint_dir="checkpoints")
history = trainer.train_loop(train_loader, val_loader, epochs=500, patience=50)
trainer.load_checkpoint("best_model.pt")
print("Best model loaded.")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history["train_loss"], label="Train"); ax1.plot(history["val_loss"], label="Val")
ax1.set(xlabel="Epoch", ylabel="Loss", title="Total Loss"); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(history["train_node"], label="Node Train"); ax2.plot(history["val_node"], label="Node Val")
ax2.plot(history["train_edge"], label="Edge Train", ls='--')
ax2.plot(history["val_edge"], label="Edge Val", ls='--')
ax2.set(xlabel="Epoch", ylabel="Loss", title="Node vs Edge Loss"); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()


---\n## 5. Parity Plots

In [ ]:
model.eval()
all_yn, all_ynp, all_ye, all_yep = [], [], [], []
with torch.no_grad():
    for batch in val_loader:
        out = model(batch)
        all_yn.append(batch["component"].y)
        all_ynp.append(out["node_preds"])
        all_ye.append(batch["component","physical","component"].y)
        all_yep.append(out["edge_preds"])

yn_t = torch.cat(all_yn).numpy(); yn_p = torch.cat(all_ynp).numpy()
ye_t = torch.cat(all_ye).numpy(); ye_p = torch.cat(all_yep).numpy()

targets = [
    ("Qubit Freq (GHz)", yn_t, yn_p, 0, 1.0),
    ("Anharmonicity (MHz)", yn_t, yn_p, 1, 100.0),
    ("Cavity Freq (GHz)", yn_t, yn_p, 2, 1.0),
    ("g (MHz)", ye_t, ye_p, 0, 100.0),
    ("Kappa (kHz)", ye_t, ye_p, 1, 100.0),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
for i, (name, yt, yp, idx, sc) in enumerate(targets):
    t, p = yt[:, idx] * sc, yp[:, idx] * sc
    axes[i].scatter(t, p, alpha=0.3, s=8, color='crimson')
    lo, hi = min(t.min(), p.min()), max(t.max(), p.max())
    if lo != hi:
        axes[i].plot([lo, hi], [lo, hi], 'k--', lw=2)
        axes[i].text(0.05, 0.9, f"R² = {r2_score(t, p):.3f}", transform=axes[i].transAxes,
                    fontsize=12, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    axes[i].set(title=name, xlabel="True", ylabel="Predicted"); axes[i].grid(alpha=0.3)
axes[-1].axis('off')
plt.suptitle("Parity Plots: Node & Edge Targets", fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()


---
## 6. Scale Invariance: The Holy Grail

The model was trained on **qubit-claw-resonator-feedline** graphs. Because it uses heterogeneous GNN on geometric embeddings, it can handle **different topologies at inference**.

### Case 1: Qubit-Claw Only (reduced topology)


In [ ]:
test_row = df.iloc[-1]
lyt_test = build_layout_from_row(test_row.to_dict())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
plot_layout(lyt_test, ax=ax1)
ax1.set_title("Full Layout (reference)", fontweight="bold")

plot_component(lyt_test["qubit"], "qubit", ax=ax2, show_etch=False)
plot_component(lyt_test["claw"], "claw", ax=ax2, show_etch=False)
ax2.autoscale_view()
ax2.set_title("Case 1: Qubit-Claw Only", fontweight="bold")
ax2.set_aspect("equal")
plt.tight_layout()
plt.show()

In [ ]:
reduced_netlist = CircuitNetlist(
    components=[
        ComponentSpec(name="qubit", component_type="TransmonCross"),
        ComponentSpec(name="claw", component_type="Claw"),
    ],
    edges=[EdgeSpec(src="qubit", dst="claw", coupling_type="capacitive")],
)

data_r = builder.build(lyt_test, reduced_netlist, global_features=GRAPH_GLOBAL_FEATURES)
model.eval()
with torch.no_grad():
    out_r = model(data_r)

summary_r = read_prediction_summary(data_r, out_r)

print("=== Case 1: Qubit-Claw Only ===")
print(
    f"Graph: {data_r['component'].x.size(0)} components, "
    f"{data_r['component','physical','component'].edge_index.size(1)//2} physical edges\n"
)

display(
    pd.DataFrame(
        [
            {
                "component_name": item.component_name,
                "component_type": item.component_type,
                "target_name": item.target_name,
                "value": item.value,
            }
            for item in summary_r.nodes
        ]
    )
)
display(
    pd.DataFrame(
        [
            {
                "component_a": item.component_a,
                "component_b": item.component_b,
                "target_name": item.target_name,
                "value": item.value,
                "directions_aggregated": item.directions_aggregated,
            }
            for item in summary_r.edges
        ]
    )
)

print(
    f"Ground truth: qubit_freq={test_row['qubit_frequency_GHz']:.3f}, "
    f"anharmonicity={test_row['anharmonicity_MHz']:.2f}, g={test_row['g_MHz']:.2f}"
)
print("Note: no cavity_freq/kappa readout since no resonator/feedline are present.")

### Case 2: Extended Topology (6 components)

Add a second feedline and resonator pair. The model automatically provides:
- `cavity_freq` for resonator2
- `kappa` for the resonator2-feedline2 edge


In [ ]:
from shapely.affinity import translate, scale as shp_scale
from shapely.ops import unary_union
from shapely.geometry import box
from squadds.ml.universal.features.node_encoder import get_polygon_for_component

# ── Build extended geometry via affine transforms ─────────────────────────────
# feedline1: x≈[-6,6], y=[1100,1300]  (center y=1200)
# feedline2: same x, shifted down 400 um → y=[700,900]
# Combined feedline: single bar from y=700 to y=1300 (looks galvanically joined)
dy = -400

def shift_comp(comp_dict, dx=0, dy=0):
    out = {}
    for k, v in comp_dict.items():
        try:
            out[k] = translate(v, xoff=dx, yoff=dy)
        except Exception:
            out[k] = v
    return out

fl_poly  = lyt_test["feedline"].get("trace") or lyt_test["feedline"].get("polygon")
fl2_poly = translate(fl_poly, yoff=dy)

# Merged feedline: union + fill gap between them → one solid bar
fl_merged = unary_union([fl_poly, fl2_poly,
                         box(fl_poly.bounds[0], fl2_poly.bounds[3],
                             fl_poly.bounds[2], fl_poly.bounds[1])])

# resonator2: shift resonator down + mirror horizontally → goes rightward
res_poly  = get_polygon_for_component(lyt_test["resonator"])
res2_poly = translate(shp_scale(res_poly, xfact=-1, yfact=1, origin=(0, 0)), yoff=dy)

# Build lyt_ext for GNN cell below
lyt_ext = dict(lyt_test)
lyt_ext["feedline1"] = lyt_test["feedline"]
lyt_ext["resonator1"] = lyt_test["resonator"]
fl2_dict = shift_comp(lyt_test["feedline"], dy=dy)
lyt_ext["feedline2"]  = fl2_dict
lyt_ext["resonator2"] = {"polygon": res2_poly,
                          "params": lyt_test["resonator"].get("params", {})}

# ── Visualization ──────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

# Left: training topology
plot_layout(lyt_test, ax=ax1)
ax1.set_title("Training Topology (4 components)\nqubit → claw → resonator → feedline",
    fontweight="bold")

# Right: extended topology
plot_component(lyt_test["qubit"],    "qubit",     ax=ax2, show_etch=False)
plot_component(lyt_test["claw"],     "claw",      ax=ax2, show_etch=False)
plot_component(lyt_test["resonator"],"resonator1",ax=ax2, show_etch=False)

# Single merged feedline bar (prime_start at top, prime_end at bottom)
ax2.fill(*fl_merged.exterior.xy, color="#9b59b6", alpha=0.85, zorder=3)
ax2.plot(*fl_merged.exterior.xy, color="#6c3483", lw=1.5, zorder=4)
# prime_start / prime_end dots
top_y  = fl_poly.bounds[3]
bot_y  = fl2_poly.bounds[1]
ax2.plot(0, top_y, "o", color="#6c3483", ms=5, zorder=5)
ax2.plot(0, bot_y, "o", color="#6c3483", ms=5, zorder=5)
ax2.text(18, top_y, "prime_start", va="center", fontsize=7, color="#6c3483")
ax2.text(18, bot_y, "prime_end",   va="center", fontsize=7, color="#6c3483")

# resonator2 (teal, mirrors resonator1 going rightward)
ax2.fill(*res2_poly.exterior.xy, alpha=0.55, color="teal", zorder=2)
ax2.plot(*res2_poly.exterior.xy, color="teal", lw=1.5, zorder=3)
cx, cy = res2_poly.centroid.x, res2_poly.centroid.y
ax2.text(cx, cy, "resonator2", ha="center", va="center", fontsize=7,
    bbox=dict(boxstyle="round,pad=0.2", fc="teal", alpha=0.8, ec="none"),
    color="white", zorder=5)

ax2.autoscale_view(); ax2.set_aspect("equal")
ax2.set_title(
    "Extended Topology (6 components)\n"
    "qubit → claw → resonator1 → feedline (upper)\n"
    "                           resonator2 ← feedline (lower)",
    fontweight="bold", fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_xlabel("x (μm)"); ax2.set_ylabel("y (μm)")
plt.tight_layout()
plt.show()


In [ ]:
ext_netlist = CircuitNetlist(
    components=[
        ComponentSpec(name="qubit", component_type="TransmonCross"),
        ComponentSpec(name="claw", component_type="Claw"),
        ComponentSpec(name="resonator1", component_type="RouteMeander"),
        ComponentSpec(name="feedline1", component_type="CoupledLineTee"),
        ComponentSpec(name="feedline2", component_type="CoupledLineTee"),
        ComponentSpec(name="resonator2", component_type="RouteMeander"),
    ],
    edges=[
        EdgeSpec(src="qubit", dst="claw", coupling_type="capacitive"),
        EdgeSpec(src="claw", dst="resonator1", coupling_type="galvanic"),
        EdgeSpec(src="resonator1", dst="feedline1", coupling_type="capacitive"),
        EdgeSpec(src="feedline1", dst="feedline2", coupling_type="galvanic"),
        EdgeSpec(src="feedline2", dst="resonator2", coupling_type="capacitive"),
    ],
)

data_ext = builder.build(lyt_ext, ext_netlist, global_features=GRAPH_GLOBAL_FEATURES)
with torch.no_grad():
    out_ext = model(data_ext)

summary_ext = read_prediction_summary(data_ext, out_ext)

print("=== Case 2: Extended Topology (6 components) ===")
print(
    f"Graph: {data_ext['component'].x.size(0)} components, "
    f"{data_ext['component','physical','component'].edge_index.size(1)//2} physical edges\n"
)

display(
    pd.DataFrame(
        [
            {
                "component_name": item.component_name,
                "component_type": item.component_type,
                "target_name": item.target_name,
                "value": item.value,
            }
            for item in summary_ext.nodes
        ]
    )
)
display(
    pd.DataFrame(
        [
            {
                "component_a": item.component_a,
                "component_b": item.component_b,
                "target_name": item.target_name,
                "value": item.value,
                "directions_aggregated": item.directions_aggregated,
            }
            for item in summary_ext.edges
        ]
    )
)

print("--- Ground Truth (from test_row) ---")
print(f"  qubit_freq_GHz  : {test_row['qubit_frequency_GHz']:.3f} GHz")
print(f"  anharmonicity   : {test_row['anharmonicity_MHz']:.2f} MHz")
print(f"  cavity_freq_GHz : {test_row['cavity_frequency_GHz']:.3f} GHz")
print(f"  g_MHz           : {test_row['g_MHz']:.2f} MHz")
print(f"  kappa_kHz       : {test_row['kappa_kHz']:.2f} kHz")
print("\nNote: ground truth applies to the training-topology geometry used to seed the example.")

In [ ]:
import networkx as nx
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe

# ── Port mapping: which port each component uses to connect ──────────────────
# (based on physical coupling geometry in our training layout)
COUPLING_PORTS = {
    ("qubit",    "claw"):     ("E",            "—"),
    ("claw",     "resonator"):("coupling_pin",  "start"),
    ("resonator","feedline"): ("end",           "coupling_pin"),
}
COUPLING_COLORS = {"capacitive": "#E91E63", "galvanic": "#FF9800"}
COUPLING_STYLE  = {"capacitive": "--",      "galvanic": "-"}

# ── Build graph with edge metadata ───────────────────────────────────────────
G = nx.DiGraph()

comp_types = {e.name: e.component_type for e in netlist.components}
for c in netlist.components:
    pins = lyt.get(c.name, {}).get("pins", {})
    G.add_node(c.name, ctype=c.component_type, pins=list(pins.keys()))

for e in netlist.edges:
    G.add_edge(e.src, e.dst, coupling=e.coupling_type)

# ── Layout: left-to-right chain ──────────────────────────────────────────────
chain = [c.name for c in netlist.components]
pos = {name: (i * 2.5, 0) for i, name in enumerate(chain)}
pos["virtual_hub"] = (len(chain) / 2 - 0.5, -1.8)
G.add_node("virtual_hub", ctype="Virtual", pins=[])
for c in netlist.components:
    G.add_edge(c.name, "virtual_hub", coupling="spatial")

fig, ax = plt.subplots(figsize=(14, 6))

# ── Draw component boxes ─────────────────────────────────────────────────────
BOX_W, BOX_H = 1.6, 0.9
node_colors = {"TransmonCross": "#1565C0", "Claw": "#E65100",
               "RouteMeander": "#2E7D32", "CoupledLineTee": "#6A1B9A",
               "Virtual": "#BF360C"}
for name, (x, y) in pos.items():
    if name == "virtual_hub": continue
    ctype = G.nodes[name]["ctype"]
    color = node_colors.get(ctype, "#555")
    rect = mpatches.FancyBboxPatch((x - BOX_W/2, y - BOX_H/2), BOX_W, BOX_H,
        boxstyle="round,pad=0.08", facecolor=color, edgecolor="white",
        linewidth=1.5, zorder=3)
    ax.add_patch(rect)
    ax.text(x, y + 0.12, name, ha="center", va="center",
        fontsize=10, fontweight="bold", color="white", zorder=4)
    ax.text(x, y - 0.18, ctype.replace("TransmonCross","Transmon\nCross"), 
        ha="center", va="center", fontsize=7, color="#ddd", style="italic", zorder=4)

    # Draw port stubs
    pins = G.nodes[name]["pins"]
    for pi, pin in enumerate(pins):
        offset = (pi - (len(pins)-1)/2) * 0.3
        # Place pin on top or bottom based on name
        py = y + BOX_H/2 + 0.1 if pin in ["N","prime_start"] else y - BOX_H/2 - 0.1
        px = x + offset
        if pin in ["E", "W"]:
            px = x + BOX_W/2 + 0.1 if pin == "E" else x - BOX_W/2 - 0.1
            py = y
        ax.plot([x + (BOX_W/2 if pin=="E" else -BOX_W/2 if pin=="W" else offset),
                 px], [y, py], color="white", lw=1.5, zorder=4)
        ax.plot(px, py, "o", ms=6, color="white", zorder=5,
                markeredgecolor=color, markeredgewidth=1.5)
        ax.text(px + (0.12 if pin in ["E","prime_end"] else -0.12 if pin=="W" else 0),
                py + (0 if pin in ["E","W"] else 0.15), pin,
                ha="center", va="bottom", fontsize=7, color="#333", zorder=6)

# ── Draw hub ─────────────────────────────────────────────────────────────────
hx, hy = pos["virtual_hub"]
hub_circle = mpatches.Circle((hx, hy), 0.55, facecolor="#BF360C",
    edgecolor="white", linewidth=1.5, zorder=3)
ax.add_patch(hub_circle)
ax.text(hx, hy + 0.08, "VIRTUAL", ha="center", va="center",
    fontsize=8, fontweight="bold", color="white", zorder=4)
ax.text(hx, hy - 0.15, "HUB", ha="center", va="center",
    fontsize=8, fontweight="bold", color="white", zorder=4)

# ── Draw physical coupling edges ─────────────────────────────────────────────
for e in netlist.edges:
    x0, y0 = pos[e.src];  x1, y1 = pos[e.dst]
    color = COUPLING_COLORS[e.coupling_type]
    ls    = COUPLING_STYLE[e.coupling_type]
    ax.annotate("", xy=(x1 - BOX_W/2, y1), xytext=(x0 + BOX_W/2, y0),
        arrowprops=dict(arrowstyle="-|>", color=color, lw=2,
            linestyle=ls, connectionstyle="arc3,rad=0.0"))
    mid_x = (x0 + x1) / 2
    ax.text(mid_x, y0 + 0.55, e.coupling_type, ha="center", va="bottom",
        fontsize=9, color=color, fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=color, alpha=0.9))

# ── Draw spatial edges to hub ─────────────────────────────────────────────────
for c in netlist.components:
    x0, y0 = pos[c.name]
    ax.annotate("", xy=(hx, hy + 0.55), xytext=(x0, y0 - BOX_H/2),
        arrowprops=dict(arrowstyle="-", color="#9E9E9E", lw=1,
            linestyle=(0,(4,4)), connectionstyle=f"arc3,rad={0.1*(chain.index(c.name)-1.5)}"))

# ── Legend ────────────────────────────────────────────────────────────────────
from matplotlib.lines import Line2D
legend_elems = [
    mpatches.Patch(facecolor="#E91E63", label="Capacitive coupling"),
    mpatches.Patch(facecolor="#FF9800", label="Galvanic coupling"),
    Line2D([0],[0], color="#9E9E9E", lw=1, linestyle=(0,(4,4)), label="Spatial → hub"),
    Line2D([0],[0], marker="o", color="w", markerfacecolor="white",
           markeredgecolor="#555", markersize=8, label="Port / pin"),
]
ax.legend(handles=legend_elems, fontsize=9, loc="upper right", framealpha=0.95)

ax.set_xlim(-1.5, len(chain)*2.5)
ax.set_ylim(-2.8, 1.5)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("Training Circuit Graph — Components, Ports & Coupling Types",
    fontsize=13, fontweight="bold", pad=12)
plt.tight_layout(); plt.show()


In [ ]:
import networkx as nx
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

# ── 1. Create the base graph ──────────────────────────────────────────────────
G = nx.Graph()

# Component node colors (matching your prior palettes)
NODE_COLORS = {"TransmonCross": "#2196F3", "Claw": "#FF9800",
               "RouteMeander": "#4CAF50", "CoupledLineTee": "#9C27B0"}
# Edge styling
COUPLING_COLORS = {"capacitive": "#E91E63", "galvanic": "#FF9800"}
COUPLING_STYLE  = {"capacitive": "dashed",  "galvanic": "solid"}

# Populate nodes
for c in netlist.components:
    targets = NODE_INFERENCE_READOUT.get(c.component_type, [])
    # Format label: Node name at top, targets in brackets below
    label_text = f"{c.name}\n" + "\n".join([f" [{t}] " for t in targets])
    G.add_node(c.name, ctype=c.component_type, label=label_text)

# Populate edges
for e in netlist.edges:
    # Identify source/dest types to lookup targets
    src_type = next(c.component_type for c in netlist.components if c.name == e.src)
    dst_type = next(c.component_type for c in netlist.components if c.name == e.dst)
    
    # Check map (handles either direction)
    targets = EDGE_INFERENCE_READOUT.get((src_type, dst_type), [])
    if not targets:
        targets = EDGE_INFERENCE_READOUT.get((dst_type, src_type), [])
        
    G.add_edge(e.src, e.dst, coupling=e.coupling_type, targets=targets)

# ── 2. Draw the graph ─────────────────────────────────────────────────────────
chain = [c.name for c in netlist.components]
# Simple linear layout
pos = {name: (i * 2.5, 0) for i, name in enumerate(chain)}

fig, ax = plt.subplots(figsize=(13, 5))

# Draw edges first (so they sit behind nodes)
for u, v, data in G.edges(data=True):
    color = COUPLING_COLORS.get(data["coupling"], "black")
    ls = COUPLING_STYLE.get(data["coupling"], "solid")
    
    ax.plot([pos[u][0], pos[v][0]], [pos[u][1], pos[v][1]], 
            color=color, linestyle=ls, lw=3.5, zorder=1)
    
    # Overlay edge Hamiltonian targets (e.g. g_MHz, kappa_kHz)
    if data["targets"]:
        mid_x = (pos[u][0] + pos[v][0]) / 2
        target_str = "\n".join([f" [{t}] " for t in data["targets"]])
        ax.text(mid_x, 0.15, target_str, ha="center", va="bottom",
                fontsize=10, fontweight="bold", color=color,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=color, alpha=0.9, lw=1.5))

# Draw nodes (circles)
for name, (x, y) in pos.items():
    ctype = G.nodes[name]["ctype"]
    color = NODE_COLORS.get(ctype, "#777")
    
    # Circle patch
    circle = mpatches.Circle((x, y), 0.35, facecolor=color, edgecolor="white", 
                             linewidth=2, zorder=2)
    ax.add_patch(circle)
    
    # Overlay node Hamiltonian targets (e.g. qubit_freq, cavity_freq)
    ax.text(x, y - 0.5, G.nodes[name]["label"], ha="center", va="top",
            fontsize=10, fontweight="bold", color="#333",
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=color, alpha=0.9, lw=1.5))

# ── 3. Legend and final touch-ups ─────────────────────────────────────────────
legend_elems = [
    Line2D([0],[0], color="#E91E63", lw=3, linestyle="dashed", label="Capacitive Coupling"),
    Line2D([0],[0], color="#FF9800", lw=3, linestyle="solid",  label="Galvanic Coupling"),
]
for ctype, color in NODE_COLORS.items():
    legend_elems.append(Line2D([0],[0], marker="o", color="w", markerfacecolor=color, 
                               markersize=12, label=ctype))

ax.legend(handles=legend_elems, fontsize=9, loc="upper right", 
          bbox_to_anchor=(1.0, 1.15), ncol=3, framealpha=0.95)

ax.set_xlim(-1, len(chain) * 2.5 - 1.5)
ax.set_ylim(-1.6, 1.2)
ax.axis("off")
ax.set_title("Hamiltonian Target Allocation in GNN Graph", 
             fontsize=14, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()


---
## Summary

| | Tabular DNN (Tutorial 8) | Universal GNN (Tutorial 12) |
|---|---|---|
| Input | Fixed parameter vector | Heterogeneous graph of geometric embeddings |
| Embedding protocol | N/A | Versioned deterministic embeddings (`EmbeddingConfig`) |
| Embedding-space science | N/A | Clustering benchmark, arithmetic benchmark, projection + similarity tools |
| Node targets | All from one output | `qubit_freq`, `anharmonicity`, `cavity_freq` per component |
| Edge targets | N/A | `g`, `kappa` per physical edge |
| Training setup | Manual feature engineering | Workflow helpers build graphs, dimensions, and structured readouts |
| New topology | Impossible | Seamless — just build the graph |
| Readout | Single output | Per-type readout map with structured summaries |